# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdulm111/ML-Assignement01/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane maps to **Ranking/Scoring**. The decision behind it is "which ones first" which
visible pages are under-capturing clicks relative to other pages in the same position tier,
and should be reviewed first. That's not a yes/no question, so classification doesn't fit.
It's also not asking what kinds of items exist, so clustering doesn't fit either. What the
editor needs is an ordered priority list, which is what ranking/scoring produces.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Abdulm111/ML-Assignement01"
REPO_DIR = "ML-Assignement01"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
visible = df[df["impressions_90d"] >= 100].copy()

print("ctr is continuous, not a yes/no label:")
print(visible["ctr"].describe()[["min", "25%", "50%", "75%", "max"]].round(4))

ctr is continuous, not a yes/no label:
min     0.00
25%     0.00
50%     0.14
75%     0.34
max    11.76
Name: ctr, dtype: float64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

I'll predict the **expected CTR** for a page  not from a fixed rule like the tier average,
but learned from real features (`content_type`, `main_intent`, `position_tier`, `word_count`).
The target I train on is the page's actual `ctr` value — a real, measured number, not something
I invented myself, so it satisfies "the target must be observed, not defined." The proxy for
review priority is `ctr_gap`: the difference between a page's actual CTR and what the model
expected for a page like it. The bigger the negative gap, the higher the priority to review.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
tier_avg = visible.groupby("position_tier")["ctr"].transform("mean")
visible["expected_ctr_tier_only"] = tier_avg
visible["ctr_gap"] = visible["ctr"] - tier_avg

print("Pages furthest below their tier's expected CTR (highest review priority):")
cols = ["position_tier", "content_type", "ctr", "expected_ctr_tier_only", "ctr_gap"]
print(visible.sort_values("ctr_gap").head(5)[cols].round(4).to_string(index=False))

Pages furthest below their tier's expected CTR (highest review priority):
position_tier       content_type  ctr  expected_ctr_tier_only  ctr_gap
       page_1 comparison article  0.0                  0.3548  -0.3548
       page_1    keyword article  0.0                  0.3548  -0.3548
       page_1    keyword article  0.0                  0.3548  -0.3548
       page_1    keyword article  0.0                  0.3548  -0.3548
       page_1    keyword article  0.0                  0.3548  -0.3548


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Since `ctr` is continuous, precision@K doesn't apply  that needs a known correct set, which
I don't have yet. The honest metric is **how much prediction error the model has compared to
a naive baseline**, checkable today: using only the tier average as the "prediction" leaves a
measurable amount of unexplained CTR variance. If adding real features (like `content_type`)
lowers that unexplained variance, that's concrete, computable-now evidence ML adds something
beyond a flat rule.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
var_tier_only = visible["ctr_gap"].var()

tier_type_avg = visible.groupby(["position_tier", "content_type"])["ctr"].transform("mean")
resid_tier_type = visible["ctr"] - tier_type_avg
var_tier_type = resid_tier_type.var()

reduction = (1 - var_tier_type / var_tier_only) * 100
print(f"Leftover CTR variance, tier-only baseline: {var_tier_only:.5f}")
print(f"Leftover CTR variance, tier+content_type:  {var_tier_type:.5f}")
print(f"Variance reduction from adding content_type: {reduction:.1f}%")

Leftover CTR variance, tier-only baseline: 0.15256
Leftover CTR variance, tier+content_type:  0.14743
Variance reduction from adding content_type: 3.4%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one page. Below is the actual slice my lane works from: position tier, content
type, intent, word count, and CTR  filtered to pages with enough impressions to trust the
CTR number.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane_slice = visible[["content_id", "position_tier", "content_type", "main_intent", "word_count", "ctr", "ctr_gap"]]
lane_slice.head(10)

,content_id,position_tier,content_type,main_intent,word_count,ctr,ctr_gap
0,content_304f48230142,striking,keyword article,transactional,3221.0,0.76,0.504218
1,content_a1fb4e703a9e,page_3_5,keyword article,informational,2481.0,0.05,-0.092359
2,content_9aa793d4d895,page_3_5,keyword article,informational,3515.0,0.09,-0.052359
3,content_331d6c4de07b,page_1,keyword article,commercial,NaN,0.49,0.135240
4,content_d99b7a2d90ca,page_3_5,keyword article,informational,2803.0,0.13,-0.012359
5,content_d4084a4bc775,page_1,keyword article,transactional,3080.0,0.03,-0.324760
7,content_a63219c6e95a,page_3_5,keyword article,commercial,NaN,0.06,-0.082359
8,content_5e6c160719bc,page_3_5,keyword article,informational,3807.0,0.09,-0.052359
9,content_c27558df2b0c,page_1,keyword article,informational,NaN,0.16,-0.194760
10,content_d8ee6cc6d642,top_3,keyword article,commercial,NaN,1.55,1.215872


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Expected CTR is continuous and the pattern behind it is real, so a single if-statement can't
capture it , you'd need to hand-encode a separate rule for every position_tier × content_type
× main_intent combination, and still miss interactions between them. The numbers below show
this isn't hypothetical: content_type alone shifts a page's expected CTR by close to half a
percentage point beyond what tier explains  bigger than some of the gaps between tiers
themselves. That's real signal a flat rule can't reach by design.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Leftover CTR gap by content_type, after removing tier effect:")
print(visible.groupby("content_type")["ctr_gap"].mean().round(4))

Leftover CTR gap by content_type, after removing tier effect:
content_type
comparison article   -0.1636
feedly article        0.4059
keyword article      -0.0039
Name: ctr_gap, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.